# 🚀 YOLOv8m (1024px) Training Pipeline for Thermal Insulation Pipe Counting

This notebook fine-tunes **YOLOv8 Medium (`yolov8m.pt`, 25.9M params)** at high resolution (**1024x1024**) on a **Google Colab T4 GPU** to push pipe detection accuracy to **98%–99%**.

---
### 📋 Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Save.
2. Upload `pipe_dataset_colab.zip` to the left files panel.
3. Run all cells below sequentially.


In [ ]:
# Step 1: Verify GPU & Install Dependencies
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime


In [ ]:
# Step 2: Unzip Dataset
import os, zipfile

os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile('/content/pipe_dataset_colab.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

print('Dataset successfully unzipped to /content/data!')
print('Train images:', len(os.listdir('/content/data/images/train')))
print('Val images:', len(os.listdir('/content/data/images/val')))


In [ ]:
# Step 3: Train YOLOv8m at imgsz=1024 on T4 GPU
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data='/content/data/data.yaml',
    epochs=50,
    imgsz=1024,
    batch=8,           # Optimal for T4 16GB VRAM
    device=0,          # CUDA Device 0
    lr0=0.0003,        # Stable initial learning rate
    lrf=0.01,          # Cosine learning rate decay
    patience=15,       # Early stopping patience
    mosaic=1.0,        # Blends 4 images into 1 for dense touching pipes
    mixup=0.15,        # Blends scenes for shadow robustness
    save=True,
    project='pipe_training',
    name='yolov8m_1024'
)


In [ ]:
# Step 4: Validate Accuracy & Print Metrics
best_model = YOLO('/content/pipe_training/yolov8m_1024/weights/best.pt')
metrics = best_model.val(data='/content/data/data.yaml', imgsz=1024)

print('\n' + '='*50)
print('🎯 YOLOv8m 1024px FINAL ACCURACY METRICS:')
print(f'   Precision:  {metrics.box.p[0]*100:.1f}%')
print(f'   Recall:     {metrics.box.r[0]*100:.1f}%')
print(f'   mAP@50:     {metrics.box.map50*100:.1f}%')
print(f'   mAP@50-95:  {metrics.box.map*100:.1f}%')
print('='*50)


In [ ]:
# Step 5: Export to Optimized ONNX at 1024px
onnx_path = best_model.export(format='onnx', imgsz=1024, optimize=True)
print(f'Successfully exported to ONNX: {onnx_path}')


In [ ]:
# Step 6: Package and Download Trained Weights
from google.colab import files
import shutil

shutil.copy('/content/pipe_training/yolov8m_1024/weights/best.pt', '/content/best_insulation_pipe_m_1024.pt')
shutil.copy('/content/pipe_training/yolov8m_1024/weights/best.onnx', '/content/best_insulation_pipe_m_1024.onnx')

print('Downloading best_insulation_pipe_m_1024.pt...')
files.download('/content/best_insulation_pipe_m_1024.pt')

print('Downloading best_insulation_pipe_m_1024.onnx...')
files.download('/content/best_insulation_pipe_m_1024.onnx')
